# Stellarator Island Chain Control

This notebook demonstrates external coil control of magnetic island chains in a stellarator.

## Scientific Background

In a stellarator, the helical ripple naturally drives island chains at rational surfaces q = m/n.
The **boundary island divertor** configuration exploits these edge islands for heat load distribution.
External coils can:

1. **Suppress** an island chain (destructive interference in ψ_mn)
2. **Phase-shift** an island chain (rotate island O-points)
3. Create **side effects** — the 按下葫芦起了瓢 (press-down-gourd, come-up-ladle) problem

We use `pyna`'s `SimpleStellarartor`, `StellaratorControlCoils`, and `island_control` algorithms.

In [ ]:
import sys
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

# Add pyna to path
sys.path.insert(0, str(Path('../../').resolve()))

from pyna.mag.stellarator import SimpleStellarartor, simple_stellarator
from pyna.mag.coil_system import StellaratorControlCoils, CoilSet, biot_savart_field
from pyna.mag.island_control import (
    island_suppression_current,
    phase_control_current,
    multi_mode_control,
    compute_resonant_amplitude,
    _natural_perturbation_func,
)
from pyna.imas_compat import (
    IMASEquilibriumIDS,
    IMASCoilsNonAxisymmetric,
    IMASPoincareMapping,
)

print('pyna loaded successfully')

## 1. Build a SimpleStellarartor with natural island chains

We choose parameters so that the q=4/4=1 and q=4/3 surfaces both lie in the plasma.

In [ ]:
# Stellarator parameters
# q profile: q0=1.1 to q1=5.0  →  q=4/3≈1.33 is near axis, q=4/2=2 at mid-radius
stella = simple_stellarator(
    R0=3.0,
    r0=0.30,
    B0=1.0,
    q0=1.1,
    q1=5.0,
    m_h=4,
    n_h=4,
    epsilon_h=0.05,  # 5% helical ripple
)
print(stella)

# Find resonant surfaces
for m, n in [(4, 4), (4, 3), (4, 2), (3, 2), (5, 4)]:
    psi_list = stella.resonant_psi(m, n)
    if psi_list:
        print(f'  q={m}/{n}={m/n:.3f}  →  psi_res={psi_list[0]:.3f}')
    else:
        print(f'  q={m}/{n}={m/n:.3f}  →  not in [0,1]')

## 2. Poincaré Map: Natural Island Chain (Boundary Island Divertor)

In [ ]:
from pyna.topo.poincare import PoincareMapper   # standard pyna Poincaré tracer

# Target island: q = 4/4 = 1  (driven by helical ripple m_h=4, n_h=4)
TARGET_M, TARGET_N = 4, 4

# Field lines near the resonant surface
start_pts = stella.start_points_near_resonance(TARGET_M, TARGET_N, n_lines=20, delta_psi=0.08)
print(f'Tracing {len(start_pts)} field lines near q={TARGET_M}/{TARGET_N}...')

# Poincaré map: use 8 workers for parallel tracing
N_TRANSITS = 200   # number of toroidal transits per line
N_WORKERS = 8

def trace_one(pt):
    mapper = PoincareMapper(stella.field_func, phi_section=0.0)
    return mapper.trace(pt, n_transits=N_TRANSITS)

with ThreadPoolExecutor(max_workers=N_WORKERS) as ex:
    results_natural = list(ex.map(trace_one, start_pts))

print('Done.')

In [ ]:
fig, ax = plt.subplots(figsize=(7, 7))
for res in results_natural:
    if res is not None and len(res) > 0:
        R_pts = np.array([p[0] for p in res])
        Z_pts = np.array([p[1] for p in res])
        ax.plot(R_pts, Z_pts, ',', color='steelblue', alpha=0.6, ms=0.8)

# Draw LCFS
theta_arr = np.linspace(0, 2*np.pi, 200)
ax.plot(stella.R0 + stella.r0 * np.cos(theta_arr),
        stella.r0 * np.sin(theta_arr), 'k--', lw=1, label='LCFS')

ax.set_xlabel('R (m)')
ax.set_ylabel('Z (m)')
ax.set_title(f'Natural Poincaré map — q={TARGET_M}/{TARGET_N} island chain\n'
             f'(Boundary Island Divertor configuration)')
ax.set_aspect('equal')
ax.legend()
plt.tight_layout()
plt.savefig('natural_island_poincare.png', dpi=120)
plt.show()

## 3. Create External Control Coils

In [ ]:
# Create a StellaratorControlCoils targeting mode (4,4)
N_COILS = 16    # number of saddle coils
R_COIL = 0.38   # slightly outside plasma (r0=0.30 m)
I0_COIL = 500.0  # reference current (A)

control_coils = StellaratorControlCoils(
    R0=stella.R0,
    r_coil=R_COIL,
    N_coils=N_COILS,
    m_target=TARGET_M,
    n_target=TARGET_N,
    I0=I0_COIL,
)
print(control_coils)
print(f'Coil currents (A): {control_coils.get_currents().round(1)}')

## 4. Island Suppression: Scan Coil Current

In [ ]:
from pyna.mag.island_control import _natural_perturbation_func, compute_resonant_amplitude
from pyna.mag.coil_system import biot_savart_field

psi_res_target = stella.resonant_psi(TARGET_M, TARGET_N)[0]
nat_func = _natural_perturbation_func(stella)

# Natural amplitude at target surface
b_nat = compute_resonant_amplitude(nat_func, psi_res_target, TARGET_M, TARGET_N, stella,
                                    n_theta=32, n_phi=32)
print(f'Natural |b̃_{TARGET_M}{TARGET_N}| = {abs(b_nat):.3e}')

# Find suppression current
print('\nFinding island suppression currents...')
# Also monitor mode (4,3) for gourd problem
I_opt, report = island_suppression_current(
    stella, control_coils,
    target_m=TARGET_M, target_n=TARGET_N,
    monitor_modes=[(4, 3), (4, 2)],
    I_max=2000.0,
    n_theta=32, n_phi=32,
)

print(f'\n=== Suppression Report ===')
print(f'  |b̃| before:  {report["target_amplitude_before"]:.3e}')
print(f'  |b̃| after:   {report["target_amplitude_after"]:.3e}')
print(f'  Suppression: {(1 - report["suppression_ratio"])*100:.1f}%')
print(f'\n=== Monitor modes (按下葫芦起了瓢 check) ===')
for mode in report['monitor_amplitudes_before']:
    b_before = report['monitor_amplitudes_before'][mode]
    b_after_ = report['monitor_amplitudes_after'][mode]
    ratio = b_after_ / (b_before + 1e-30)
    print(f'  q={mode[0]}/{mode[1]}: |b̃| {b_before:.3e} → {b_after_:.3e}  (×{ratio:.2f})')

In [ ]:
# Scan coil amplitude to show island width vs current
def add_coil_field(R, Z, phi, coil_set):
    """Get total coil field at a point."""
    R_arr = np.array([[R]])
    Z_arr = np.array([[Z]])
    phi_arr = np.array([[phi]])
    br = bz = bp = 0.0
    for pts, I in coil_set.coils:
        _br, _bz, _bp = biot_savart_field(pts, I, R_arr, Z_arr, phi_arr)
        br += float(_br); bz += float(_bz); bp += float(_bp)
    return br, bz, bp

I0_scan = np.linspace(0, 1000, 15)
b_total_scan = []

# Linear model: b_total(alpha) = b_nat + alpha * (b_coil_unit)
b_coil_unit = compute_resonant_amplitude(
    lambda R, Z, phi: add_coil_field(R, Z, phi,
        type('_', (), {'coils': [(pts, I/I0_COIL) for pts, I in control_coils.coils]})()),
    psi_res_target, TARGET_M, TARGET_N, stella, n_theta=24, n_phi=24
)

for I0 in I0_scan:
    b_tot = b_nat + b_coil_unit * I0
    b_total_scan.append(abs(b_tot))

plt.figure(figsize=(8, 4))
plt.plot(I0_scan, np.array(b_total_scan) / abs(b_nat), 'b-o', ms=5)
plt.axhline(0, color='gray', ls='--')
plt.xlabel('Control coil current $I_0$ (A)')
plt.ylabel('$|\\tilde{b}_{mn}| / |\\tilde{b}_{mn,nat}|$')
plt.title(f'Island suppression scan: mode ({TARGET_M},{TARGET_N})')
plt.grid(True)
plt.tight_layout()
plt.savefig('island_suppression_scan.png', dpi=120)
plt.show()

## 5. 按下葫芦起了瓢: Press-Down-Gourd Problem

As we suppress the (4,4) island, we monitor what happens to other island chains.

In [ ]:
# For each I0 in scan, compute amplitude at target and monitor surfaces
# Monitor: (4,3)
MONITOR_M, MONITOR_N = 4, 3
psi_res_monitor = stella.resonant_psi(MONITOR_M, MONITOR_N)

if psi_res_monitor:
    psi_res_mon = psi_res_monitor[0]
    b_nat_mon = compute_resonant_amplitude(
        nat_func, psi_res_mon, MONITOR_M, MONITOR_N, stella, n_theta=24, n_phi=24)
    
    # Response of monitor mode to unit coil current
    b_coil_mon = compute_resonant_amplitude(
        lambda R, Z, phi: add_coil_field(R, Z, phi,
            type('_', (), {'coils': [(pts, I/I0_COIL) for pts, I in control_coils.coils]})()),
        psi_res_mon, MONITOR_M, MONITOR_N, stella, n_theta=24, n_phi=24
    )
    
    b_target_scan = [abs(b_nat + b_coil_unit * I0) for I0 in I0_scan]
    b_monitor_scan = [abs(b_nat_mon + b_coil_mon * I0) for I0 in I0_scan]
    
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(8, 6), sharex=True)
    
    ax1.plot(I0_scan, np.array(b_target_scan)/abs(b_nat), 'b-o', ms=5, label=f'Target ({TARGET_M},{TARGET_N})')
    ax1.set_ylabel('Normalized $|\\tilde{b}_{mn}|$')
    ax1.axhline(1, color='gray', ls=':', lw=0.8)
    ax1.legend()
    ax1.grid(True)
    ax1.set_title('按下葫芦起了瓢: Island control side effects')
    
    ax2.plot(I0_scan, np.array(b_monitor_scan)/abs(b_nat_mon), 'r-s', ms=5, label=f'Monitor ({MONITOR_M},{MONITOR_N})')
    ax2.set_xlabel('Control coil current $I_0$ (A)')
    ax2.set_ylabel('Normalized $|\\tilde{b}_{mn}|$')
    ax2.axhline(1, color='gray', ls=':', lw=0.8)
    ax2.legend()
    ax2.grid(True)
    
    plt.tight_layout()
    plt.savefig('gourd_problem.png', dpi=120)
    plt.show()
else:
    print(f'Monitor surface q={MONITOR_M}/{MONITOR_N} not in plasma')

## 6. Phase Control: Rotating Island O-Points

In [ ]:
# Demonstrate phase control: shift island O-points by pi/4
phase_shifts = [0, np.pi/4, np.pi/2, 3*np.pi/4]

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes_flat = axes.ravel()

for idx, dphase in enumerate(phase_shifts):
    ax = axes_flat[idx]
    
    # Build a modified field function with phase-shifted perturbation
    # Approximate: shift coil phase to rotate island
    import copy
    cc_phase = copy.deepcopy(control_coils)
    
    # Reset to find phase-control currents
    cc_phase2 = StellaratorControlCoils(
        R0=stella.R0, r_coil=R_COIL, N_coils=N_COILS,
        m_target=TARGET_M, n_target=TARGET_N, I0=I0_COIL,
    )
    I_phase = phase_control_current(
        stella, cc_phase2,
        target_m=TARGET_M, target_n=TARGET_N,
        desired_phase_shift=dphase,
        I_max=1500.0, n_theta=24, n_phi=24,
    )
    
    # Build perturbed field function
    def make_modified_field(coil_set, base_stella):
        """Return field_func that adds external coil field to stellarator field."""
        def field_func(rzphi_1d):
            R, Z, phi = rzphi_1d
            # Base field
            base = base_stella.field_func(rzphi_1d)
            # Coil perturbation (only BR component matters significantly)
            R_arr = np.array([[R]])
            Z_arr = np.array([[Z]])
            phi_arr = np.array([[phi]])
            dBR = dBZ = dBP = 0.0
            for pts, I in coil_set.coils:
                _br, _bz, _bp = biot_savart_field(pts, I, R_arr, Z_arr, phi_arr)
                dBR += float(_br); dBZ += float(_bz); dBP += float(_bp)
            # Add perturbation to total field then renormalize
            theta_ = np.arctan2(Z, R - base_stella.R0)
            psi_ = base_stella.psi_ax(R, Z)
            q_ = float(base_stella.q_of_psi(psi_))
            r_minor_ = np.sqrt((R - base_stella.R0)**2 + Z**2)
            B_phi_ = base_stella.B0 * base_stella.R0 / R
            B_pol_ = B_phi_ * r_minor_ / (R * max(abs(q_), 1e-3))
            if r_minor_ > 1e-10:
                BR0_ = -B_pol_ * np.sin(theta_)
                BZ0_ = B_pol_ * np.cos(theta_)
            else:
                BR0_ = BZ0_ = 0.0
            dBR_nat = base_stella.epsilon_h * base_stella.B0 * psi_ * np.cos(
                base_stella.m_h * theta_ - base_stella.n_h * float(phi))
            BR_t = BR0_ + dBR_nat + dBR
            BZ_t = BZ0_ + dBZ
            BP_t = B_phi_ + dBP
            Bmag = np.sqrt(BR_t**2 + BZ_t**2 + BP_t**2) + 1e-30
            return np.array([BR_t/Bmag, BZ_t/Bmag, BP_t/(R*Bmag)])
        return field_func
    
    field_mod = make_modified_field(cc_phase2, stella)
    
    # Trace Poincaré map
    def trace_mod(pt):
        mapper = PoincareMapper(field_mod, phi_section=0.0)
        return mapper.trace(pt, n_transits=150)
    
    with ThreadPoolExecutor(max_workers=N_WORKERS) as ex:
        results_phase = list(ex.map(trace_mod, start_pts))
    
    for res in results_phase:
        if res is not None and len(res) > 0:
            R_pts = np.array([p[0] for p in res])
            Z_pts = np.array([p[1] for p in res])
            ax.plot(R_pts, Z_pts, ',', color='steelblue', alpha=0.6, ms=0.8)
    
    ax.plot(stella.R0 + stella.r0*np.cos(theta_arr),
            stella.r0*np.sin(theta_arr), 'k--', lw=0.8)
    ax.set_title(f'Phase shift Δφ = {dphase/np.pi:.2f}π')
    ax.set_aspect('equal')
    ax.set_xlabel('R (m)')
    ax.set_ylabel('Z (m)')

fig.suptitle('Island Phase Control: Rotating O-Points', fontsize=13)
plt.tight_layout()
plt.savefig('phase_control.png', dpi=120)
plt.show()

## 7. Export Data in IMAS-Compatible JSON Format

In [ ]:
import os
os.makedirs('imas_output', exist_ok=True)

# Export equilibrium IDS
eq_ids = IMASEquilibriumIDS.from_stellarator(stella, n_psi=64, n_theta=128)
eq_ids.to_json('imas_output/equilibrium_ids.json')
print('Exported equilibrium_ids.json')
print(f'  R0={eq_ids.r0} m, B0={eq_ids.b0} T')
print(f'  q range: [{eq_ids.q.min():.2f}, {eq_ids.q.max():.2f}]')

# Export coils IDS
coils_ids = IMASCoilsNonAxisymmetric.from_coil_set(control_coils)
coils_ids.to_json('imas_output/coils_non_axisymmetric_ids.json')
print(f'Exported coils_non_axisymmetric_ids.json ({len(coils_ids.coil_names)} coils)')

# Export Poincaré map as IDS
all_R = []
all_Z = []
for res in results_natural:
    if res is not None:
        for p in res:
            all_R.append(p[0])
            all_Z.append(p[1])

poincare_ids = IMASPoincareMapping.from_poincare_data(
    np.array(all_R), np.array(all_Z), phi_section=0.0)
poincare_ids.to_json('imas_output/poincare_mapping_ids.json')
print(f'Exported poincare_mapping_ids.json ({len(all_R)} crossings)')

In [ ]:
# Verify round-trip
eq_ids2 = IMASEquilibriumIDS.from_json('imas_output/equilibrium_ids.json')
coils_ids2 = IMASCoilsNonAxisymmetric.from_json('imas_output/coils_non_axisymmetric_ids.json')
poincare_ids2 = IMASPoincareMapping.from_json('imas_output/poincare_mapping_ids.json')

assert np.allclose(eq_ids2.q, eq_ids.q), 'Equilibrium round-trip failed'
assert len(coils_ids2.coil_names) == len(coils_ids.coil_names), 'Coils round-trip failed'
print('✓ All IMAS IDS round-trips verified')

## 8. Summary

| Capability | Description |
|---|---|
| `SimpleStellarartor` | Analytic helical-ripple stellarator with linear q profile |
| `StellaratorControlCoils` | Saddle coil array phased for (m,n) resonant control |
| `biot_savart_field` | Parallelized Biot-Savart on cylindrical grids |
| `island_suppression_current` | Optimal currents to suppress island chain |
| `phase_control_current` | Rotate island O-points by desired phase angle |
| `multi_mode_control` | Joint suppression of multiple modes (gourd problem) |
| `IMASEquilibriumIDS` | IMAS-compatible equilibrium export/import |
| `IMASCoilsNonAxisymmetric` | IMAS-compatible coil geometry export |
| `IMASPoincareMapping` | IMAS-compatible Poincaré section export |

The **按下葫芦起了瓢** (press-down-gourd) problem is visible: suppressing (4,4) can amplify (4,3). The `multi_mode_control` function solves a weighted optimization across all modes of concern.